<a href="https://colab.research.google.com/github/protogia/formula1-evaluations/blob/main/gp_brazil_2025_review.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Prologue

What a weekend! The 2025 GP Brazil delivered on its promise of adrenaline-fueled action, with unexpected twists and turns across both qualifying and race events. This year, Sao Paulo played host to a thrilling Sprint Race, adding another layer of intensity to an already legendary circuit.

In this notebook, we'll dive deep into the performance metrics of the Sprint Qualifying, Sprint Race, Main Qualifying, and the Grand Prix itself. As highlighted in my [preview for this race](https://protogia.github.io/blog/formula1-gp-brazil-preview-2025/), this circuit is notorious for its dramatic elevation changes, boasting gradients from a steep -15% downhill to a challenging 15% uphill. These undulations aren't just scenic; they're a significant factor in car setup and driver strategy. The upcoming plot vividly illustrates the circuit's layout, complete with crucial corner annotations and precise gradient information, setting the stage for our performance analysis.

In [6]:
!pip install fastf1
!pip install git+https://github.com/protogia/formula-1-plotly-utils.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.0/136.0 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 427.6/427.6 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.6 MB/s eta 0:00:00
  Attempting uninstall: msgpack
    Found existing installation: msgpack 1.2.0
    Uninstalling msgpack-1.2.0:
      Successfully uninstalled msgpack-1.2.0
  Cloning https://github.com/protogia/formula-1-plotly-utils.git to /tmp/pip-req-build-w1cp878b
  Running command git clone --filter=blob:none --quiet https://github.com/protogia/formula-1-plotly-utils.git /tmp/pip-req-build-w1cp878b
  Resolved https://github.com/protogia/formula-1-plotly-utils.git to commit c6e3272986bf68a2e3fb4e889e27dc750d0a5fac
  Installing build de

In [7]:
# log-config
import warnings
warnings.filterwarnings('ignore')

# layout-config
from IPython.core import display
display.display_html(display.HTML(""))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [8]:
# load data
import fastf1
from formula_1_plotly_utils import utils

SQ = fastf1.get_session(2025, 'Brazil', 'SQ')
SQ.load()

req         WARNING 	DEFAULT CACHE ENABLED! (24.0 KB) /root/.cache/fastf1
core           INFO 	Loading data for São Paulo Grand Prix - Sprint Qualifying [v3.8.3]
INFO:fastf1.fastf1.core:Loading data for São Paulo Grand Prix - Sprint Qualifying [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
INFO:fastf1.api:Fetching session info data...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
INFO:fastf1.api:Fetching driver list...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
core        WARNING 	Sprint Qualifying is n

In [9]:
position = SQ.laps.pick_fastest().get_pos_data()
circuit_info = SQ.get_circuit_info()
reference_altitude = 800

fig = utils.plot_track(
    position=position,
    circuit_info=circuit_info,
    reference_altitude=reference_altitude,
)

fig.show()

## Sprint Qualifying

The Sprint Qualifying session unfolded under clear, dry skies, with no rain impacting the track before the action began. However, as an evening event, temperatures were a critical factor. The air temperature averaged a cool 19°C, but what truly stands out in the weather data is the **dramatic drop in track temperature**, plummeting from 43°C down to 34°C. This significant cooling, coupled with minimal wind speeds around 0.7m/s during Phase 2, likely had a profound effect on tire grip and overall car balance. The decreasing track temperature, exacerbated by the progressive elimination of 10 drivers, created an evolving challenge for those fighting for pole position, as we'll visualize in the weather plot below.

In [10]:
fig = utils.plot_weather_data(SQ.weather_data)
fig.show()

### Tyre Strategy: The Medium-Soft Gamble

Given the dry conditions, tire strategies in Sprint Qualifying were under the spotlight. As the next chart vividly illustrates, most drivers initially opted for the medium compound, aiming to maximize their track time and understand the evolving conditions. However, a crucial strategic shift occurred in Qualifying Round 3: nearly all contenders swapped to the soft compound in their final two laps. This aggressive move was a clear bid to unlock extra pace and secure the best possible grid position for the Sprint Race.

Fernando Alonso, notably, was one of the first to commit to soft tires, perhaps sensing a lack of grip. Despite this early switch, he couldn't quite break into the top 10. Intriguingly, Lando Norris and Lewis Hamilton were the only drivers to bravely attempt Qualifying Round 3 on medium compounds, a decision that ultimately prevented them from advancing. This highlights the fine margins and bold gambles that define F1 qualifying, where tire choice can make or break a session.

In [11]:
drivers = SQ.laps['Driver'].unique()

fig = utils.plot_tyre_strategies(
    drivers=drivers,
    laps=SQ.laps,
    track_status=SQ.track_status,
)
fig.show()

### Lap Time Dynamics: Unpacking the Performance

The subsequent charts offer a detailed look at lap time distribution, first by Qualifying Round (Q1, Q2, Q3) and then by tire compound. While the raw distribution might appear varied due to differing car performances and driver efforts, key insights emerge.

Perhaps the most fascinating detail is the observation that the median lap times of drivers eliminated early in qualifying were actually *lower* than those who ultimately secured pole position. This paradox isn't about raw pace, but rather strategy: top drivers often achieve their best laps with fewer attempts, demonstrating efficiency and precision, while others might push harder, generating more laps, in a desperate bid to find pace.

As expected, the soft compound tires generally delivered superior performance. However, this advantage was often unlocked only during specific, high-effort 'push' laps, indicating that drivers were carefully managing their tire performance throughout the session rather than consistently extracting maximum pace. This nuanced interplay of strategy and raw speed defines the competitive landscape of Sprint Qualifying.

In [13]:
fig = utils.plot_laptime_distribution_per_qualifyinground(
    drivers=drivers,
    laps=SQ.laps,
    results=SQ.results,
)

fig.show()

In [17]:
# filter out unwanted lap types (e.g., pit laps)
all_laps = SQ.laps.pick_quicklaps().reset_index()
all_laps['LapTimeSeconds'] = all_laps['LapTime'].dt.total_seconds()

# drivers sorted by final position in the sprint qualifying
if not SQ.results.empty:
    driver_positions = SQ.results.sort_values(by='Position')['Abbreviation'].tolist()
    all_laps = all_laps[all_laps['Driver'].isin(driver_positions)].copy()
    all_laps['Driver_Category'] = pd.Categorical(all_laps['Driver'], categories=driver_positions, ordered=True)
    all_laps.sort_values(by='Driver_Category', inplace=True)
else:
    driver_positions = all_laps['Driver'].unique() # Use all drivers with laps if results are not available


# box plot compounds
fig_box_compound = px.box(all_laps,
                          x='Driver',
                          y='LapTimeSeconds',
                          color='Compound',
                          points='all',
                          hover_data=['LapNumber'],
                          title='Lap Time Performance per Driver and Tyre Compound (Sprint Qualifying)')

fig_box_compound.update_layout(
    xaxis_title='Driver',
    yaxis_title='Lap Time (seconds)',
    legend_title='Tyre Compound',
    xaxis=dict(categoryorder='array', categoryarray=driver_positions)
)

fig_box_compound.show()

### The Best of the Best: Official Lap Times

The next chart presents the definitive best lap times, categorized by Qualifying Round and sorted by overall fastest performance. This visualization dramatically highlights strategic successes and missed opportunities. Fernando Alonso's situation is particularly noteworthy: despite recording the second-fastest lap overall across all qualifying phases, his inability to replicate that blistering pace in the critical Qualifying Round 3 meant he started the Sprint Race from a respectable, but ultimately less advantageous, fifth position. This underscores how crucial consistency and peak performance in the final stages of qualifying are for grid placement, even for drivers capable of immense speed.

In [18]:
# Q1, Q2, Q3 columns to seconds
best_lap_times_official = SQ.results[['Abbreviation', 'Q1', 'Q2', 'Q3']].copy()
for col in ['Q1', 'Q2', 'Q3']:
    best_lap_times_official[col] = best_lap_times_official[col].apply(lambda x: x.total_seconds() if pd.notna(x) else np.nan)

best_lap_times_official = best_lap_times_official.melt(
    id_vars='Abbreviation',
    value_vars=['Q1', 'Q2', 'Q3'],
    var_name='QualifyingRound',
    value_name='BestLapTime'
).dropna(subset=['BestLapTime']) # drop rows with NaN best lap times

# best overall lap time per driver from the official results for sorting
best_overall_lap_time_driver_official = best_lap_times_official.groupby('Abbreviation')['BestLapTime'].min().reset_index()
best_overall_lap_time_driver_official = best_overall_lap_time_driver_official.rename(columns={'BestLapTime': 'BestOverallLapTime'})

# merge best lap times with overall best lap time for sorting
best_lap_times_official = pd.merge(best_lap_times_official, best_overall_lap_time_driver_official, on='Abbreviation', how='left')

if not best_overall_lap_time_driver_official.empty:
    driver_order_official = best_overall_lap_time_driver_official.sort_values(by='BestOverallLapTime')['Abbreviation'].tolist()
    best_lap_times_official['Driver_Category'] = pd.Categorical(best_lap_times_official['Abbreviation'], categories=driver_order_official, ordered=True)
    best_lap_times_official.sort_values(by='Driver_Category', inplace=True)


# scatter plot best lap times
fig_scatter = px.scatter(best_lap_times_official,
                         x='Abbreviation',
                         y='BestLapTime',
                         color='QualifyingRound', # color by Qualifying Round
                         symbol='QualifyingRound',
                         hover_data=['QualifyingRound', 'BestLapTime'],
                         title='Best Lap Time per Driver by Qualifying Round')

fig_scatter.update_layout(
    xaxis_title='Driver',
    yaxis_title='Best Lap Time (seconds)',
    legend_title='Qualifying Round',
    xaxis=dict(categoryorder='array', categoryarray=driver_order_official) # Set the order of drivers on the x-axis
)

fig_scatter.show()

## The Sprint Race

### Sprint Race Conditions: Rising Temperatures and Unexpected Drama

With the brevity of Sprint Qualifying offering limited insight into long-run car and driver performance, the 24-lap Sprint Race took center stage just a day later, mere hours before the main Qualifying session. The weather remained dry, but a subtle yet significant shift occurred: air temperatures climbed by 4 degrees compared to the previous day, while track temperatures settled between a warmer 25°C and 29°C. These warmer conditions could influence tire degradation and optimal car setup.

However, as the subsequent weather plot reveals, the real drama unfolded with track events. These conditions, combined with the unfolding race action, set the stage for crucial strategic decisions, particularly regarding tire management.

In [19]:
SR = fastf1.get_session(2025, "Brazil", "S")
SR.load()

core           INFO 	Loading data for São Paulo Grand Prix - Sprint [v3.8.3]
INFO:fastf1.fastf1.core:Loading data for São Paulo Grand Prix - Sprint [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
INFO:fastf1.api:Fetching session info data...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
INFO:fastf1.api:Fetching driver list...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_

In [23]:
fig = utils.plot_weather_data(SR.weather_data)
fig.show()

### Sprint Race Tyre Strategies: Adapting to Chaos

The next chart dissects the intricate tire strategies deployed during the Sprint Race. A defining moment for many drivers was the red flag phase, triggered by the unfortunate dropouts of Colapinto and Piastri. This disruption provided a critical window for tire changes, a strategic opportunity many seized.

Initially, six drivers (excluding Stroll) bravely started on the soft compound, aiming for early pace, while the majority (fourteen drivers) opted for the more durable medium tires. As the race unfolded, especially after the red flag, we see a fascinating split: six drivers who started on mediums transitioned to softs, capitalizing on the interruption, while another six steadfastly remained on their initial medium compounds. This dynamic interplay of pre-race planning and reactive strategy, influenced heavily by the race-altering red flag, is vividly captured in the visualization below.

In [24]:
drivers = SR.laps['Driver'].unique()

fig = utils.plot_tyre_strategies(
    drivers=drivers,
    laps=SR.laps,
    track_status=SR.track_status,
)
fig.show()

### Performance Under Pressure: Soft vs. Medium in the Sprint

The violin plot below offers a compelling look at how tire choices translated into lap time performance during the Sprint Race. Among the top 5 finishers, only Lando Norris demonstrated exceptional pace predominantly on medium compound tires, showcasing his car's remarkable balance and his driving skill. In stark contrast, nearly all other drivers who committed solely to medium tires (with the notable exception of Lewis Hamilton) struggled, finishing outside the points in 11th position or worse. This data strongly suggests a significant performance differential: drivers who did not utilize the soft compound at some point in the race faced a median lap time disadvantage of at least one second. This highlights the crucial role of the softer compound in achieving competitive lap times, especially when the race is interrupted and strategy can be adapted.

In [25]:
# filter out unwanted lap types (e.g., pit laps)
race_laps = SR.laps.pick_quicklaps().reset_index()
race_laps['LapTimeSeconds'] = race_laps['LapTime'].dt.total_seconds()

# drivers sorted by final position in the sprint race
if not SR.results.empty:
    driver_order_sprint_race = SR.results.sort_values(by='Position')['Abbreviation'].tolist()
    race_laps = race_laps[race_laps['Driver'].isin(driver_order_sprint_race)].copy()
    race_laps['Driver_Category'] = pd.Categorical(race_laps['Driver'], categories=driver_order_sprint_race, ordered=True)
    race_laps.sort_values(by='Driver_Category', inplace=True)
else:
    driver_order_sprint_race = race_laps['Driver'].unique() # Use all drivers with laps if results are not available


# violin plot best lap times
fig_violin_sprint = px.violin(race_laps,
                             x='Driver',
                             y='LapTimeSeconds',
                             color='Compound',
                             box=True,
                             points='all',
                             hover_data=['LapNumber', 'Compound', 'LapTimeSeconds'],
                             title='Sprint Race Lap Time Distribution per Driver and Tyre Compound')

fig_violin_sprint.update_layout(
    xaxis_title='Driver',
    yaxis_title='Lap Time (seconds)',
    legend_title='Tyre Compound',
    xaxis=dict(categoryorder='array', categoryarray=driver_order_sprint_race) # Set the order of drivers on the x-axis
)

fig_violin_sprint.show()

### Overtakes and Position Battles: Who Gained, Who Lost?

Despite the early retirements of Piastri and Colapinto, the Sprint Race saw surprisingly stable positions for many drivers. However, the plot of driver positions per lap reveals some standout performances in the midfield. Oliver Bearman, for instance, delivered a masterclass in overtaking, climbing from an 18th-place start to finish an impressive 12th – marking the highest number of overtakes in the race. On the flip side, Hulkenberg experienced a challenging race, losing the most positions as he dropped from 10th to 18th. These individual battles, though not always at the very front, demonstrate the relentless fight for every position on the challenging Brazilian circuit.

In [26]:
fig = go.Figure()

for driver in drivers:
    drv_laps = SR.laps.pick_drivers(driver)

    if not drv_laps.empty:
        abb = drv_laps['Driver'].iloc[0]
        fig.add_trace(go.Scatter(
            x=drv_laps['LapNumber'],
            y=drv_laps['Position'],
            mode='lines+markers',
            name=abb,
            hoverinfo='text',
            text=[f'Driver: {abb}<br>Lap: {lap}<br>Position: {pos}' for lap, pos in zip(drv_laps['LapNumber'], drv_laps['Position'])]
        ))

fig.update_layout(
    title='Driver Positions Per Lap',
    xaxis_title='Lap Number',
    yaxis_title='Position',
    yaxis=dict(
        autorange='reversed', # P1 at the top
    ),
    legend_title='Driver'
)

fig.show()

## Main Qualifying

### Qualifying Weather: A Different Climate for Pole Position

The weather conditions for the main Qualifying session presented a slightly different challenge compared to the Sprint events. The track remained resolutely dry, with no rain threatening the crucial push for pole. Air temperatures hovered comfortably around 25°C. However, the track temperature experienced another notable decline, starting at a blistering 47°C but cooling down to 36°C by the session's end. Wind speeds remained relatively low, fluctuating between 0.6m/s and 2.8m/s. This gradual cooling of the track as the session progressed likely influenced tire performance and grip, demanding adaptability from teams and drivers as they chased the fastest lap times.

In [27]:
Q = fastf1.get_session(2025, 'Brazil', 'Q')
Q.load()

core           INFO 	Loading data for São Paulo Grand Prix - Qualifying [v3.8.3]
INFO:fastf1.fastf1.core:Loading data for São Paulo Grand Prix - Qualifying [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
INFO:fastf1.api:Fetching session info data...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
INFO:fastf1.api:Fetching driver list...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
INFO:fastf1.fastf1.req:No cached data found for 

In [30]:
fig = utils.plot_weather_data(Q.weather_data)
fig.show()

### Qualifying Tyre Strategy: Soft Domination and Gasly's Maverick Move

The tire strategy in the main Qualifying session was largely dominated by the soft compound, as expected. With the exception of just seven drivers, every competitor completed their qualifying efforts exclusively on soft tires, underscoring its performance advantage in a single-lap shootout.

However, Pierre Gasly emerged as a strategic outlier. He began his qualifying with soft tires but uniquely transitioned to mediums, a bold and unusual choice for a session where every millisecond counts. This decision, clearly visible in the tire strategy chart below, distinguishes him from the rest of the field. Fortunately, despite the intense competition, there were no mechanical retirements, allowing all drivers to push their cars to the limit.

In [36]:
drivers = Q.laps['Driver'].unique()

fig = utils.plot_tyre_strategies(
    drivers=drivers,
    laps=Q.laps,
    track_status=Q.track_status,
)
fig.show()

### Best Lap Times: A Tale of Two Qualifyings

Moving beyond overall lap time distributions, which can be noisy, our focus shifts to the definitive best lap times achieved in each Qualifying Round. The scatter plot below reveals a fascinating trend: the majority of drivers extracted their absolute best performance in Qualifying Round 2, suggesting optimal track conditions and driver rhythm at that stage.

Crucially, when comparing these results to the Sprint Qualifying, a stark difference emerges: only Lando Norris managed to replicate a similarly blistering lap time. Most other top-10 contenders found themselves up to 500ms slower than their Sprint Qualifying bests. Yet, in a remarkable display of progress, Oliver Bearman defied this trend, improving his best lap time by approximately 200ms. This comparison highlights the dynamic nature of conditions and driver adaptation between the two qualifying formats, with Norris's consistency and Bearman's advancement being particularly noteworthy.

In [32]:
# Q1, Q2, Q3 columns to seconds
best_lap_times_official = Q.results[['Abbreviation', 'Q1', 'Q2', 'Q3']].copy()
for col in ['Q1', 'Q2', 'Q3']:
    best_lap_times_official[col] = best_lap_times_official[col].apply(lambda x: x.total_seconds() if pd.notna(x) else np.nan)

best_lap_times_official = best_lap_times_official.melt(
    id_vars='Abbreviation',
    value_vars=['Q1', 'Q2', 'Q3'],
    var_name='QualifyingRound',
    value_name='BestLapTime'
).dropna(subset=['BestLapTime']) # drop rows with NaN best lap times

# best overall lap time per driver from the official results for sorting
best_overall_lap_time_driver_official = best_lap_times_official.groupby('Abbreviation')['BestLapTime'].min().reset_index()
best_overall_lap_time_driver_official = best_overall_lap_time_driver_official.rename(columns={'BestLapTime': 'BestOverallLapTime'})

# merge best lap times with overall best lap time for sorting
best_lap_times_official = pd.merge(best_lap_times_official, best_overall_lap_time_driver_official, on='Abbreviation', how='left')

if not best_overall_lap_time_driver_official.empty:
    driver_order_official = best_overall_lap_time_driver_official.sort_values(by='BestOverallLapTime')['Abbreviation'].tolist()
    best_lap_times_official['Driver_Category'] = pd.Categorical(best_lap_times_official['Abbreviation'], categories=driver_order_official, ordered=True)
    best_lap_times_official.sort_values(by='Driver_Category', inplace=True)


# scatter plot best lap times
fig_scatter = px.scatter(best_lap_times_official,
                         x='Abbreviation',
                         y='BestLapTime',
                         color='QualifyingRound', # color by Qualifying Round
                         symbol='QualifyingRound',
                         hover_data=['QualifyingRound', 'BestLapTime'],
                         title='Best Lap Time per Driver by Qualifying Round')

fig_scatter.update_layout(
    xaxis_title='Driver',
    yaxis_title='Best Lap Time (seconds)',
    legend_title='Qualifying Round',
    xaxis=dict(categoryorder='array', categoryarray=driver_order_official) # Set the order of drivers on the x-axis
)

fig_scatter.show()

## The Grand Prix Race

In [34]:
GP = fastf1.get_session(2025, 'Brazil', 'R')
GP.load()

core           INFO 	Loading data for São Paulo Grand Prix - Race [v3.8.3]
INFO:fastf1.fastf1.core:Loading data for São Paulo Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
INFO:fastf1.api:Fetching session info data...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
INFO:fastf1.api:Fetching driver list...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_stat

### Race Day Conditions: A Hot Afternoon Battle

The Grand Prix race unfolded under significantly warmer conditions compared to the previous sessions, demanding a different approach to car setup and tire management. Air temperatures peaked at a comfortable 27°C, but the track temperature soared to a blistering 46°C, creating an intense challenge for tire degradation. Wind speeds remained moderate, generally between 0.8m/s and 2.5m/s.

The weather plot below vividly illustrates these race day conditions. The high track temperatures put a premium on managing the tires, as overheating could lead to rapid performance drops. This elevation in temperature, a stark contrast to the cooler Sprint Qualifying, set the stage for a physically demanding race where tire strategy would be paramount.

In [35]:
fig = utils.plot_weather_data(GP.weather_data)
fig.show()

### Grand Prix Tyre Strategies: The Dance of Durability and Speed

The 71-lap Grand Prix was a true test of endurance and strategic foresight, especially concerning tire choices. The next chart lays bare the diverse and intricate tire strategies deployed by each team, with most drivers opting for a two-stop strategy, a common choice at Interlagos due to its demanding corners and high-speed sections leading to significant tire wear.

Notice the prevalence of medium and hard compounds. The soft compound, while offering blistering pace, typically has a shorter lifespan, making its usage a calculated risk. Drivers often started on mediums, transitioned to hards for a longer stint, and then perhaps back to mediums or even softs for a final push, depending on race circumstances and safety car interventions. The plot reveals which drivers gambled on a more aggressive soft-tire strategy early on, and who opted for durability with hard compounds to extend their stints. Analyzing these patterns can offer crucial insights into each team's understanding of tire degradation and their confidence in their car's long-run pace.

In [37]:
drivers = GP.laps['Driver'].unique()

fig = utils.plot_tyre_strategies(
    drivers=drivers,
    laps=GP.laps,
    track_status=GP.track_status,
)
fig.show()

### Race Pace Analysis: Who Mastered Interlagos?

The violin plot below provides a comprehensive overview of lap time performance across the entire Grand Prix, broken down by driver and tire compound. This granular analysis is crucial for understanding who truly mastered the demanding conditions of Interlagos over a full race distance.

Observe the spread of lap times for each driver. A tighter distribution indicates greater consistency, a hallmark of strong race pace and effective tire management. Compare the median lap times across different compounds – do the softer tires show a clear pace advantage, or does their shorter life lead to a wider variance in lap times due to degradation? Pay close attention to drivers who managed to maintain competitive lap times on harder compounds, as this often signifies exceptional car balance and driving skill. This visualization will help us identify standout performances and highlight any significant strategic trade-offs made during the race.

In [38]:
# filter out unwanted lap types (e.g., pit laps)
race_laps = GP.laps.pick_quicklaps().reset_index()
race_laps['LapTimeSeconds'] = race_laps['LapTime'].dt.total_seconds()

# drivers sorted by final position in the race
if not GP.results.empty:
    driver_order_race = GP.results.sort_values(by='Position')['Abbreviation'].tolist()
    race_laps = race_laps[race_laps['Driver'].isin(driver_order_race)].copy()
    race_laps['Driver_Category'] = pd.Categorical(race_laps['Driver'], categories=driver_order_race, ordered=True)
    race_laps.sort_values(by='Driver_Category', inplace=True)
else:
    driver_order_race = race_laps['Driver'].unique() # Use all drivers with laps if results are not available


# violin plot best lap times
fig_violin_race = px.violin(race_laps,
                             x='Driver',
                             y='LapTimeSeconds',
                             color='Compound',
                             box=True,
                             points='all',
                             hover_data=['LapNumber', 'Compound', 'LapTimeSeconds'],
                             title='Grand Prix Lap Time Distribution per Driver and Tyre Compound')

fig_violin_race.update_layout(
    xaxis_title='Driver',
    yaxis_title='Lap Time (seconds)',
    legend_title='Tyre Compound',
    xaxis=dict(categoryorder='array', categoryarray=driver_order_race) # Set the order of drivers on the x-axis
)

fig_violin_race.show()

### The Race Story Unfolds: Position Changes Throughout the Grand Prix

The most compelling narrative of any Grand Prix is the ebb and flow of positions. The following plot, charting each driver's position per lap, is a real-time drama unfolding on the tarmac. Here, we can identify key overtakes, strategic undercuts or overcuts in the pit lane, and drivers battling tooth and nail for every single point.

Look for significant jumps or drops in position – these often correlate with pit stops, safety car periods that bunch up the field, or exceptional driving allowing a driver to scythe through the pack. Which drivers made the most progress from their starting grid slot? Who struggled to maintain their position? This visualization is a powerful tool for dissecting the race action, revealing the strategic brilliance and the unfortunate setbacks that define a Grand Prix.

In [39]:
fig = go.Figure()

for driver in drivers:
    drv_laps = GP.laps.pick_drivers(driver)

    if not drv_laps.empty:
        abb = drv_laps['Driver'].iloc[0]
        fig.add_trace(go.Scatter(
            x=drv_laps['LapNumber'],
            y=drv_laps['Position'],
            mode='lines+markers',
            name=abb,
            hoverinfo='text',
            text=[f'Driver: {abb}<br>Lap: {lap}<br>Position: {pos}' for lap, pos in zip(drv_laps['LapNumber'], drv_laps['Position'])]
        ))

fig.update_layout(
    title='Driver Positions Per Lap (Grand Prix Race)',
    xaxis_title='Lap Number',
    yaxis_title='Position',
    yaxis=dict(
        autorange='reversed', # P1 at the top
    ),
    legend_title='Driver'
)

fig.show()

In [41]:
import pandas as pd

# Filter for point scorers (Top 10)
points_finishers = GP.results.sort_values(by='Position').head(10)['Abbreviation'].tolist()
points_laps = GP.laps.pick_quicklaps().loc[GP.laps['Driver'].isin(points_finishers)].copy()
points_laps['LapTimeSeconds'] = points_laps['LapTime'].dt.total_seconds()

# Calculate consistency (Standard Deviation of lap times) and Average Pace
performance_stats = points_laps.groupby('Driver').agg(
    AvgLapTime=('LapTimeSeconds', 'mean'),
    MedianLapTime=('LapTimeSeconds', 'median'),
    Consistency=('LapTimeSeconds', 'std'),
    FastestLap=('LapTimeSeconds', 'min')
).reset_index()

# Merge with final positions for context
performance_summary = pd.merge(
    performance_stats,
    GP.results[['Abbreviation', 'Position', 'Status']],
    left_on='Driver',
    right_on='Abbreviation'
).sort_values(by='Position')

# Calculate gap to winner's average pace
winner_avg = performance_summary.iloc[0]['AvgLapTime']
performance_summary['GapToWinner_Avg'] = performance_summary['AvgLapTime'] - winner_avg

# Use the display function correctly
from IPython.display import display as ipy_display
ipy_display(performance_summary[['Position', 'Driver', 'AvgLapTime', 'Consistency', 'FastestLap', 'GapToWinner_Avg']])

,Position,Driver,AvgLapTime,Consistency,FastestLap,GapToWinner_Avg
6,1.0,NOR,74.145333,0.573162,73.040,0.000000
0,2.0,ANT,74.287167,0.698611,73.123,0.141833
9,3.0,VER,74.108644,0.899648,72.447,-0.036689
8,4.0,RUS,74.372203,0.781321,73.097,0.226870
7,5.0,PIA,74.230567,0.878201,72.742,0.085233
1,6.0,BEA,74.537864,0.641063,73.483,0.392531
5,7.0,LAW,75.327919,0.625117,74.029,1.182586
3,8.0,HAD,74.971000,0.688880,73.694,0.825667
4,9.0,HUL,75.267918,0.729471,73.474,1.122585
2,10.0,GAS,74.964508,0.664793,73.736,0.819175


## Conclusion

Looking back at the 71 laps of the Brazilian Grand Prix, the data paints a picture of a race won through technical mastery rather than just raw horsepower. Several key factors stand out that defined the final classification:

*   **The 'Norris Standard' of Consistency:** While the timing screens occasionally showed flashes of brilliance from Verstappen and Piastri, Lando Norris's victory was built on his low variance. His ability to maintain a 74.1s average with the lowest standard deviation in the top 10 proves that the McLaren-Norris pairing has mastered the art of managing Pirelli's thermal sensitivity in high-heat (46°C track) conditions.
*   **Midfield Efficiency:** The performance of Oliver Bearman and Liam Lawson deserves high praise. Their median lap times were often indistinguishable from the top-five runners. This suggests that the 'performance ceiling' of the midfield has been reached; for these drivers, the difference between a podium and a P7 finish is now dictated almost entirely by qualifying position and strategic execution during Safety Car windows.
*   **The Tyre Management Paradox:** Our analysis of the 'Gap to Winner' shows that even a minor loss in tire management—like we saw with Nico Hulkenberg toward the end of his second stint—compounded into a significant 1.1s average deficit. At Interlagos, if you aren't managing the 'thermal deg' in Sector 2, you are a sitting duck on the climb to the finish line.
*   **Strategic Agility:** This weekend rewarded those who could adapt. From the high-stakes tire swaps in Sprint Qualifying to the two-stop endurance test of the Grand Prix, the teams that succeeded were those that didn't just have a 'Plan A,' but had the data-driven confidence to pivot when the track temperatures began to drop.

**Final Verdict:** Interlagos remains a driver's circuit, but in 2025, it was the engineers' mastery of tire data that provided the platform for that driver brilliance to shine. A clinical weekend for the points-scorers, and a masterclass in consistency from the winner.